# 4K → 1080p converter

**No downloads to your computer.** Runs on Google's machines and works straight on the Drive folder.

1. Menu **Runtime → Change runtime type → T4 GPU → Save** (faster, optional).
2. Press ▶ on block 1 and allow access to your Google account.
3. Paste the Drive folder link into block 2 and press ▶.

For each video bigger than 1080p it uploads a 1080p copy next to it and moves the 4K original into a **4K originals** subfolder. Videos already 1080p or smaller are skipped. Then name the files in Shoot Renamer.

---

**ES:** Menú **Runtime → Change runtime type → T4 GPU → Save**. Pulsa ▶ en el bloque 1 y da acceso a tu cuenta de Google. Pega el enlace de la carpeta de Drive en el bloque 2 y pulsa ▶. Cada vídeo más grande que 1080p se convierte: la copia 1080p queda en la carpeta y el original 4K se mueve a la subcarpeta **4K originals**.

In [ ]:
# Block 1 / Bloque 1: sign in to Google Drive
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
drive = build('drive', 'v3')
me = drive.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
import subprocess
gpu = bool(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip())
print('Signed in as', me, '| GPU:', 'yes' if gpu else 'no (slower, still works)')

In [ ]:
# Block 2 / Bloque 2: paste the Drive folder link between the quotes, then press ▶
FOLDER_LINK = ''

INCLUDE_SUBFOLDERS = False   # True = also convert videos inside subfolders
QUALITY = 18                 # 18 = near-original quality, 23 = smaller files

import re, os, json, time, subprocess
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

assert FOLDER_LINK.strip(), 'Paste the folder link into FOLDER_LINK first / Pega el enlace en FOLDER_LINK'
m = re.search(r'folders/([A-Za-z0-9_-]+)', FOLDER_LINK)
root_id = m.group(1) if m else FOLDER_LINK.strip().split('?')[0]
KW = dict(supportsAllDrives=True)
LW = dict(supportsAllDrives=True, includeItemsFromAllDrives=True)
FOLDER_MIME = 'application/vnd.google-apps.folder'

def children(fid):
    out, tok = [], None
    while True:
        r = drive.files().list(q=f"'{fid}' in parents and trashed=false", pageSize=1000, pageToken=tok,
              fields='nextPageToken,files(id,name,mimeType,parents,videoMediaMetadata(width,height))', **LW).execute()
        out += r.get('files', []); tok = r.get('nextPageToken')
        if not tok: return out

def originals_folder(parent):
    for f in children(parent):
        if f['mimeType'] == FOLDER_MIME and f['name'] == '4K originals': return f['id']
    return drive.files().create(body={'name': '4K originals', 'mimeType': FOLDER_MIME, 'parents': [parent]}, fields='id', **KW).execute()['id']

def probe(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
        'stream=width,height:stream_tags=rotate:stream_side_data=rotation', '-of', 'json', path], capture_output=True, text=True)
    s = json.loads(r.stdout)['streams'][0]; w, h = int(s['width']), int(s['height'])
    rot = int(s.get('tags', {}).get('rotate', 0) or 0)
    for sd in s.get('side_data_list', []) or []:
        if 'rotation' in sd: rot = int(sd['rotation'])
    return (h, w) if abs(rot) % 180 == 90 else (w, h)

def new_name(name):
    stem = os.path.splitext(name)[0]
    m = re.match(r'^(\d{4}_.+?_.+?_.+?_\d{2,3})_(3840|2160|4K|4k|3840px|2160px)$', stem)
    return (m.group(1) if m else stem + '') + '_1080p.mp4'

def convert(src, dst):
    vf = "scale='if(gt(iw,ih),1920,-2)':'if(gt(iw,ih),-2,1920)'"
    venc = ['-c:v', 'h264_nvenc', '-preset', 'p5', '-rc', 'vbr', '-cq', str(QUALITY + 1), '-b:v', '0'] if gpu \
        else ['-c:v', 'libx264', '-preset', 'medium', '-crf', str(QUALITY)]
    return subprocess.run(['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-i', src, '-vf', vf, *venc,
        '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart', dst], capture_output=True, text=True)

def walk(fid, path):
    for f in children(fid):
        if f['mimeType'] == FOLDER_MIME:
            if INCLUDE_SUBFOLDERS and f['name'] != '4K originals': yield from walk(f['id'], path + '/' + f['name'])
        elif f['mimeType'].startswith('video/'):
            yield f, fid, path

root_name = drive.files().get(fileId=root_id, fields='name', **KW).execute()['name']
videos = list(walk(root_id, root_name))
print(f'{len(videos)} video(s) in "{root_name}"\n')
os.makedirs('/content/work', exist_ok=True)
done = skipped = failed = 0
for f, parent, path in videos:
    vm = f.get('videoMediaMetadata') or {}
    if vm.get('width') and max(vm['width'], vm['height']) <= 1920:
        print(f'  = {f["name"]}  already {vm["width"]}x{vm["height"]}, skipped'); skipped += 1; continue
    t = time.time(); src = '/content/work/src' + os.path.splitext(f['name'])[1]; dst = '/content/work/out.mp4'
    try:
        print(f'  > {f["name"]}  downloading…', end=' ', flush=True)
        with open(src, 'wb') as fh:
            dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=f['id'], **KW), chunksize=64 * 1024 * 1024)
            while not dl.next_chunk()[1]: pass
        w, h = probe(src)
        if max(w, h) <= 1920:
            print(f'already {w}x{h}, skipped'); skipped += 1; continue
        print(f'{w}x{h} converting…', end=' ', flush=True)
        r = convert(src, dst)
        if r.returncode != 0: raise RuntimeError(r.stderr[-300:])
        print('uploading…', end=' ', flush=True)
        name = new_name(f['name'])
        drive.files().create(body={'name': name, 'parents': [parent]},
            media_body=MediaFileUpload(dst, mimetype='video/mp4', resumable=True, chunksize=64 * 1024 * 1024), fields='id', **KW).execute()
        keep = originals_folder(parent)
        drive.files().update(fileId=f['id'], addParents=keep, removeParents=parent, fields='id', **KW).execute()
        print(f'✓ {name} ({time.time() - t:.0f}s)'); done += 1
    except Exception as e:
        print('FAILED:', str(e)[:300]); failed += 1
    finally:
        for p in (src, dst):
            if os.path.exists(p): os.remove(p)
print(f'\nConverted {done} · skipped {skipped} · failed {failed}')
if done: print('Now name them in Shoot Renamer: https://jordialbanell.github.io/shoot-renamer/')